## tl;dr

PN18 recursively paired every lower prime child into one product parent, paired local p29-wheel candidates into a
second tree, and used GCD as their relation. At the fresh anchor `700,000,000,000`, it sealed `+9` before target
primality was checked. Independent validation confirmed `700,000,000,009` as the first prime above the anchor and
passed `36/36` checks.

The recursion is exact but not a genuine compression or speed improvement. The child root is a 1,205,845-bit
integer, and the full construction was slower and larger than efficient established controls.


In [1]:
from pathlib import Path
import hashlib, json

HERE = Path(r'F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes')
prediction = json.loads((HERE / 'PN18_RECURSIVE_TEARA_PRODUCT_TREE_PREDICTION.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN18_RECURSIVE_TEARA_PRODUCT_TREE_VALIDATION.json').read_text(encoding='utf-8'))
cost = json.loads((HERE / 'PN18_COST_AUDIT.json').read_text(encoding='utf-8'))
target = prediction['target']
print('Sealed candidate:', f"{target['predicted_integer']:,}")
print('Correction:', target['correction'])
print('Independent first-prime validation:', validation['candidate_is_first_prime_above_anchor'])
print('Checks:', validation['passed_count'], '/', validation['check_count'])


Sealed candidate: 700,000,000,009
Correction: 9
Independent first-prime validation: True
Checks: 36 / 36


## Context & Methods

### Key Assumptions

- The exact prime ridge is the absence of every prime-factor child through the square-root boundary.
- Multiplying distinct lower prime children is mathematically lossless by unique factorization, although decoding the
  factors from the product is not cheap.
- A branch GCD greater than one proves that at least one candidate collides; it cannot certify that every leaf is
  composite, so unresolved branches must be descended.
- The mathematics is established primorial/product-tree/batch-GCD arithmetic expressed as recursive ARA parents.

For lower children `q`, candidate branch `I`, anchor `N` and fixed window `W`, PN18 uses

`G = product(q <= sqrt(N+W-1))`, `M_I = product(N+t for t in I)`, and `R_I = gcd(G,M_I)`.
At a leaf, `R=1` is the exact quiet factor ridge.


In [2]:
print('Development integrity')
print('anchor | children | correction | prediction | exact')
for row in prediction['development']:
    print(f"{row['anchor']:>15,} | {row['child_count']:>8,} | {row['correction']:>10} | "
          f"{row['predicted_integer']:>15,} | {row['matches_development_control']}")


Development integrity
anchor | children | correction | prediction | exact
    100,000,000 |    1,229 |          7 |     100,000,007 | True
  1,000,000,000 |    3,401 |          7 |   1,000,000,007 | True
 10,000,000,000 |    9,592 |         19 |  10,000,000,019 | True
100,000,000,000 |   27,293 |          3 | 100,000,000,003 | True
400,000,000,000 |   51,526 |         19 | 400,000,000,019 | True


## Data

The source is exact integer arithmetic. The fresh block contains 65,536 offsets. Its complete square-root inventory
contains 66,650 prime children through gate 836,657. The primary prediction packet and child-product root were
hashed before independent target validation.


In [3]:
fields = [
    ('anchor', target['anchor']),
    ('block end', target['block_end']),
    ('sqrt boundary', target['sqrt_block_end_floor']),
    ('child count', target['child_count']),
    ('child root bits', target['child_root_bit_length']),
    ('p29 candidates in window', target['candidate_count_in_window_after_p29']),
    ('candidate tree nodes', target['candidate_tree_nodes']),
    ('GCD nodes visited', target['query']['gcd_nodes_visited']),
    ('explicit leaves queried', target['query']['explicit_candidate_leaves_queried']),
]
for label, value in fields:
    print(f'{label}: {value:,}' if isinstance(value, int) else f'{label}: {value}')
print('Child-root SHA-256:', target['child_root_sha256'])


anchor: 700,000,000,000
block end: 700,000,065,535
sqrt boundary: 836,660
child count: 66,650
child root bits: 1,205,845
p29 candidates in window: 10,349
candidate tree nodes: 20,697
GCD nodes visited: 17
explicit leaves queried: 3
Child-root SHA-256: 98ECD397C66F43A42BCC5BB6E72053A40F480D64DD7E4798A3DEF8663F7C48BF


## Results

The first two p29-wheel leaves, offsets `+1` and `+3`, shared lower-prime factors with the child parent. Offset `+9`
had GCD one and was sealed. Direct root-GCD, independent segmented sieve, deterministic Miller-Rabin and full trial
division all returned the same answer.


In [4]:
print('Result')
print('correction:', target['correction'])
print('candidate:', f"{target['predicted_integer']:,}")
print('p29 rank:', target['p29_candidate_rank_through_prediction'])
print('odd candidates through answer:', target['odd_scan_candidates_through_prediction'])
print('direct GCD reconstruction:', validation['independent_direct_root_gcd_correction'])
print('segmented-sieve reconstruction:', validation['independent_segmented_sieve_correction'])
print('first prime:', validation['candidate_is_first_prime_above_anchor'])

print('\nInformation sizes (bytes)')
for label, value in [
    ('child product root', target['child_root_byte_length']),
    ('uint32 child list', target['child_list_uint32_bytes']),
    ('one-bit odd sieve', target['one_bit_odd_sieve_bytes']),
    ('PN17-sized collision field', target['pn17_collision_field_bytes']),
    ('candidate-tree ideal payload', target['candidate_tree_transient_payload_bytes_ceil']),
]:
    print(f'{label:32s} {value:>10,}')

print('\nPost-target median implementation seconds')
for name, record in cost['results'].items():
    print(f"{name:48s} {record['median_seconds']:.9f}")


Result
correction: 9
candidate: 700,000,000,009
p29 rank: 3
odd candidates through answer: 5
direct GCD reconstruction: 9
segmented-sieve reconstruction: 9
first prime: True

Information sizes (bytes)
child product root                  150,731
uint32 child list                   266,600
one-bit odd sieve                    52,292
PN17-sized collision field          131,072
candidate-tree ideal payload        735,283

Post-target median implementation seconds
segmented_sieve_from_scratch                     0.115319400
pn18_recursive_tree_from_scratch                 1.856962600
sequential_root_gcd_with_prebuilt_root           0.002269100
p29_wheel_deterministic_miller_rabin             0.000088800


## Takeaways

1. Recursive child-to-parent pairing preserved the exact PN17 quiet ridge on five opened anchors and one fresh
   anchor.
2. The fresh `+9` result was sealed before target primality and independently validated.
3. The child product is a reusable operational parent, and GCD is an exact informative relation between that parent
   and a candidate identity.
4. “One integer” is not a low-dimensional state here: the root contains 1,205,845 bits, is larger than a one-bit
   sieve and PN17's collision field, and the candidate tree adds about 735 KB of ideal payload.
5. The present construction is an exact ARA crosswalk of established product-tree/batch-GCD primality mathematics,
   not a new prime theorem or faster search algorithm.
6. The original frozen validator had a receipt-only JSON failure on the giant integer. The unchanged prediction was
   validated under a hashed v1.1 serialization amendment, passing 36/36 checks.
